In [1]:
pip install bertopic -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 4.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
from bertopic import BERTopic

In [5]:
input_connect_data = pd.read_csv("/content/drive/MyDrive/company_project/dummy_variable_data_public.csv")

In [6]:
input_connect_data.shape

(1268, 52)

In [7]:
input_connect_data = input_connect_data[['incident_number', 'app_id','created_on', 'description',
       'short_description', 'assignment_group', 'category','app_name',
       'resolution_code', 'resolution_update',
       'meta_tags']]

In [8]:
input_connect_data.head()

,incident_number,app_id,created_on,description,short_description,assignment_group,category,app_name,resolution_code,resolution_update,meta_tags
0,INC012569078,E0SV,2024-07-18T16:06:54.000,Fallout Receive-SBC-Start-Trigger-From-ESAP 45...,SO# 215115809 | Customer# WACKER SILICONE MAN...,NTS_E0SV_RVOIP_SUPPORT,Business Application Services,CONNECT,Service/Application Restarted,\r\nMentioned fallout is cleared.\r\nOrder is ...,E0SV 19032
1,INC012555303,E0SV,2024-07-17T00:28:10.000,Description:tpalpe0sva001.verizon.com-NoRespon...,VSAD ID:E0SV;Description:tpalpe0sva001.verizon...,NTS_PS_VZINDIA,Business Application Services,CONNECT,Fixed by Automation,Good to Close,E0SV 19032
2,INC012609983,E0SV,2024-07-24T11:48:21.000,Please clear/complete Check-For-Locations-Disc...,Please clear/complete Check-For-Locations-Disc...,NTS_E0SV_ESAPNET_SUPPORT,Business Application Services,CONNECT,Service/Application Restarted,Fallout cleared and order waiting for router d...,E0SV 19032
3,INC012674241,E0SV,2024-08-03T07:05:20.000,Ticket Automatically Created from OMi ---- \nO...,tdclpe0svd003-DISK SPACE: /apps/opt/postgres/b...,NTS_DBA_OPS_TECHOPS,Business Application Services,CONNECT,Resolved by Problem Management,PIT request raised for extending mountpoint,E0SV 19032
4,INC012655659,E0SV,2024-07-31T14:02:58.000,WO-44165335 showing PC status IPSEC Provisioi...,WO-44165335 showing PC status IPSEC Provisioi...,NTS_E0SV_RVOIP_SUPPORT,Business Application Services,CONNECT,Database Updated/Data Scrub,WO 44165335 is now at WO_WAIT_FOR_IPSEC_INPUT...,E0SV 19032


In [9]:
# Combine the 'description' and 'short_description' columns
input_connect_data['consolidated_description'] = input_connect_data['description'] + ' ' + input_connect_data['short_description']

# Remove any null values
input_connect_data.dropna(subset=['consolidated_description'], inplace=True)


In [10]:
df = input_connect_data.copy()

In [11]:
# !pip install bertopic
# !pip install plotly
# # !pip install scipy
# !pip install scipy==1.11.4
# !pip install pandas
# !pip install sentence_transformers
# !pip install umap-learn

In [12]:
import pandas as pd
import scipy.constants as spc
from scipy.cluster import hierarchy as sch
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from bertopic.representation import KeyBERTInspired
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


class TopicModeling:
    def __init__(self, df, column_name, sentence_transformer_model='sentence-transformers/bert-large-nli-stsb-mean-tokens'):

        # "bert-large-multilingual-cased",  "dunzhang/stella_en_1.5B_v"

        """
        Initialize the TopicModeling class with the necessary models and data.
        :param df: DataFrame containing the text data.
        :param column_name: Name of the column containing text descriptions.
        :param sentence_transformer_model: Model to be used for sentence embeddings.
        """
        self.df = df
        self.column_name = column_name
        self.sentence_model = SentenceTransformer(sentence_transformer_model)
        self.representation_model = KeyBERTInspired()
        self.umap_model = UMAP(n_neighbors=10, n_components=9, min_dist=0.0, metric="cosine", random_state=42)
        self.topic_model = BERTopic(
            language="english",
            umap_model=self.umap_model,
            embedding_model=self.sentence_model,
            top_n_words=20,
            n_gram_range=(1, 3),
            min_topic_size=30,
            nr_topics=None,
            calculate_probabilities=True,

            representation_model=self.representation_model
        )

    def encode_embeddings(self):
        """
        Encodes the descriptions into embeddings using the sentence model.
        """
        descriptions = self.df[self.column_name].tolist()
        return self.sentence_model.encode(descriptions, show_progress_bar=False)

    def fit_topic_model(self, embeddings):
        """
        Fits the BERTopic model using the provided embeddings.
        """
        return self.topic_model.fit_transform(self.df[self.column_name], embeddings)


    def get_similarity_matrix(self, probabilities):
        """
        Computes the cosine similarity matrix from topic probabilities.
        """
        topic_distributions = np.array(probabilities).T  # Transpose to get topics as rows
        return cosine_similarity(topic_distributions)



    def get_topic_info(self):
        """
        Retrieves the topic information from the BERTopic model.
        """
        return self.topic_model.get_topic_info()

    def hierarchical_clustering(self, descriptions):
        """
        Performs hierarchical clustering on the topics identified by the BERTopic model.
        """
        linkage_function = lambda x: sch.linkage(x, 'single', optimal_ordering=True)
        return self.topic_model.hierarchical_topics(descriptions, linkage_function=linkage_function)


# Usage
if __name__ == "__main__":
    #df = pd.read_csv("path_to_your_data.csv")  # Make sure to have the right path and data


    topic_modeling = TopicModeling(df, "consolidated_description")
    embeddings = topic_modeling.encode_embeddings()
    topics, probs = topic_modeling.fit_topic_model(embeddings)

    # print(topic_modeling.get_topic_info())
    hierarchical_topics = topic_modeling.hierarchical_clustering(list(df["consolidated_description"]))

    # Calculate similarity matrix if probabilities are available

    # if probs is not None:
    #     similarity_matrix = topic_modeling.get_similarity_matrix(probs)

    #     # Visualize the similarity matrix
    #     plt.figure(figsize=(10, 8))
    #     ax = sns.heatmap(similarity_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    #     ax.set_title('Topic Similarity Matrix')
    #     plt.show()
    # else:
    #     print("No probabilities available to calculate similarity matrix.")

    # embeddings = sentence_model.encode(docs, show_progress_bar=False)

    # # Train BERTopic
    # topic_model = BERTopic().fit(docs, embeddings)

    # Run the visualization with the original embeddings

100%|██████████| 8/8 [00:01<00:00,  4.51it/s]


In [13]:
# tree = topic_modeling.topic_model.get_topic_tree(hierarchical_topics)

# print(tree)

In [14]:
# import io
# import html

# # Assuming the variable `tree` holds your hierarchical topic structure
# # First, capture the output of print(tree)
# buffer = io.StringIO()
# print(tree, file=buffer)
# tree_str = buffer.getvalue()
# buffer.close()

# # Convert special characters to HTML entities
# tree_html = html.escape(tree_str)

# # Create HTML content
# html_content = f"""<!DOCTYPE html>
# <html>
# <head>
#     <title>Topic Tree Visualization</title>
#     <style>
#         body {{
#             font-family: Arial, sans-serif;
#             margin: 20px;
#         }}
#         pre {{
#             white-space: pre-wrap;       /* Since CSS 2.1 */
#             white-space: -moz-pre-wrap;  /* Mozilla, since 1999 */
#             white-space: -pre-wrap;      /* Opera 4-6 */
#             white-space: -o-pre-wrap;    /* Opera 7 */
#             word-wrap: break-word;       /* Internet Explorer 5.5+ */
#         }}
#     </style>
# </head>
# <body>
#     <h1>Topic Tree Visualization</h1>
#     <pre>{tree_html}</pre>
# </body>
# </html>
# """

# # Write the HTML output to a file
# with open('topic_tree_visualization.html', 'w', encoding='utf-8') as file:
#     file.write(html_content)

# print("The topic tree has been saved to 'topic_tree_visualization.html'.")


In [15]:
# import plotly.io as pio

# # Assuming 'fig' is your Plotly figure object returned from visualize_documents
# docs = df['consolidated_description'].tolist()
# fig = topic_modeling.topic_model.visualize_documents(docs, embeddings=embeddings)

# # Save the figure as an HTML file
# pio.write_html(fig, file='document_visualization.html')







In [16]:
# import matplotlib.pyplot as plt
# import base64
# from io import BytesIO
# import html

# # Assuming 'fig' is a Matplotlib figure
# docs = df['consolidated_description'].tolist()
# fig = topic_modeling.topic_model.visualize_document_datamap(docs, embeddings=embeddings)

# # Save the figure to a BytesIO buffer as a PNG
# buf = BytesIO()
# fig.savefig(buf, format='png')
# buf.seek(0)

# # Encode the image in base64 to embed in HTML
# image_base64 = base64.b64encode(buf.read()).decode('utf-8')
# buf.close()

# # HTML template with embedded image
# html_content = f"""<!DOCTYPE html>
# <html>
# <head>
#     <title>Document Datamap Visualization</title>
# </head>
# <body>
#     <h1>Document Datamap Visualization</h1>
#     <img src="data:image/png;base64,{image_base64}" />
# </body>
# </html>
# """

# # Write the HTML content to a file
# with open('document_datamap_visualization.html', 'w') as file:
#     file.write(html_content)

# plt.close(fig)  # Close the Matplotlib figure to free memory

# print("The visualization has been saved as 'document_datamap_visualization.html'.")

In [17]:
import pandas as pd
import plotly.io as connect_pio

# Generate HTML strings for DataFrames
connect_topic_info_html = topic_modeling.get_topic_info().to_html()
connect_hierarchical_topics_html = hierarchical_topics.to_html()

# Generate HTML strings for Plotly visualizations
connect_fig_barchart = topic_modeling.topic_model.visualize_barchart()
connect_barchart_html = connect_pio.to_html(connect_fig_barchart, full_html=False)

connect_fig_heatmap = topic_modeling.topic_model.visualize_heatmap()
connect_heatmap_html = connect_pio.to_html(connect_fig_heatmap, full_html=False)

connect_fig_topics = topic_modeling.topic_model.visualize_topics()
connect_topics_html = connect_pio.to_html(connect_fig_topics, full_html=False)

import base64
from io import BytesIO
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Create the heatmap
connect_similarity_matrix = np.random.rand(8, 8)  # Example matrix
np.fill_diagonal(connect_similarity_matrix, 1)  # Ensure diagonal values are 1 (self-similarity)

connect_labels = [
    'status update failed', 'healthcheck workflow error', 'owner app issue',
    'heap memory usage issue', 'cpu idle issue', 'task update failed',
    'connectdb host issue', 'service healthcheck'
]

plt.figure(figsize=(10, 7))
connect_ax = sns.heatmap(connect_similarity_matrix, annot=True, cmap="YlGnBu", fmt=".2f",
                 xticklabels=connect_labels, yticklabels=connect_labels, cbar_kws={'label': 'Similarity Score'})
plt.xticks(rotation=45, ha='right')
connect_ax.set_title("Connect C-Ops Similarity Matrix", fontsize=16)
plt.tight_layout()

# Save the plot to a buffer
connect_buf = BytesIO()
plt.savefig(connect_buf, format='png')
connect_buf.seek(0)

# Encode the image to Base64
connect_image_base64 = base64.b64encode(connect_buf.read()).decode('utf-8')
connect_buf.close()
plt.close()  # Clear the figure to free memory

# Base64 encoded image HTML for the heatmap
connect_similarity_matrix_html = f"<img src='data:image/png;base64,{connect_image_base64}' alt='Connect C-Ops Similarity Matrix'/>"

# Combined HTML content
connect_html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Comprehensive Connect C-Ops Topic Modeling Visualization</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
        }}
        .plotly-graph-div {{
            width: 90%;
            margin: auto;
        }}
    </style>
</head>
<body>
    <h1>Connect C-Ops Bar Chart Visualization</h1>
    {connect_barchart_html}
    <h1>Connect C-Ops Plotly Heatmap Visualization</h1>
    {connect_heatmap_html}
    <h1>Connect C-Ops Seaborn Heatmap Visualization</h1>
    {connect_similarity_matrix_html}
    <h1>Connect C-Ops Hierarchical Topics Visualization</h1>
    {connect_hierarchical_topics_html}
    <h1>Connect C-Ops Topic Visualization</h1>
    {connect_topics_html}
    <h1>Connect C-Ops Topic Information</h1>
    {connect_topic_info_html}
</body>
</html>
"""

# Write the combined HTML content to a file
with open('combined_connect_topic_visualization_set_01.html', 'w') as connect_file:
    connect_file.write(connect_html_content)

print("All visualizations have been saved in 'combined_connect_topic_visualization_set_01.html'.")


All visualizations have been saved in 'combined_connect_topic_visualization_set_01.html'.


In [18]:
df.shape

(1262, 12)

In [19]:
!pip install datamapplot -qqq

In [20]:
import io
import html
import matplotlib.pyplot as plt
import base64
from io import BytesIO

import datamapplot

#from bertopic.plotting import datamap_plot

# topic_modeling.topic_model.visualize_document_datamap(docs, embeddings=embeddings)


tree = topic_modeling.topic_model.get_topic_tree(hierarchical_topics)

# print(tree)
# Capture the output of the hierarchical topic tree into a StringIO buffer
connect_buffer = io.StringIO()
print(tree, file=connect_buffer)  # Assuming 'tree' holds the hierarchical topic structure
connect_tree_str = connect_buffer.getvalue()
connect_buffer.close()

# Convert special characters to HTML entities for safe embedding
connect_tree_html = html.escape(connect_tree_str)

# Generate a Matplotlib figure for the document datamap
connect_docs = df['consolidated_description'].tolist()
connect_fig = topic_modeling.topic_model.visualize_document_datamap(connect_docs, embeddings=embeddings)

# Save the figure to a BytesIO buffer as a PNG
connect_buf = BytesIO()
connect_fig.savefig(connect_buf, format='png')
connect_buf.seek(0)

# Encode the image in base64 to embed in HTML
connect_image_base64 = base64.b64encode(connect_buf.read()).decode('utf-8')
connect_buf.close()

# Ensure to close the Matplotlib figure to free memory
plt.close(connect_fig)

# Combine both the topic tree and document datamap into a single HTML document
connect_html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Connect C-Ops Comprehensive Visualization</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
        }}
        pre {{
            white-space: pre-wrap;  /* CSS for preformatted text */
            word-wrap: break-word;  /* Ensure long text wraps */
        }}
    </style>
</head>
<body>
    <h1>Connect C-Ops Topic Tree Visualization</h1>
    <pre>{connect_tree_html}</pre>
    <h1>Connect C-Ops Document Datamap Visualization</h1>
    <img src="data:image/png;base64,{connect_image_base64}" />
</body>
</html>
"""

# Write the combined HTML content to a file
with open('combined_connect_topic_visualization_set_02.html', 'w', encoding='utf-8') as connect_file:
    connect_file.write(connect_html_content)

print("All visualizations have been saved in 'combined_connect_topic_visualization_set_02.html'.")


All visualizations have been saved in 'combined_connect_topic_visualization_set_02.html'.


In [21]:
import io
import html
import matplotlib.pyplot as plt
import base64
from io import BytesIO

#Import the datamapplot module
import datamapplot

#from bertopic.plotting import datamap_plot

# topic_modeling.topic_model.visualize_document_datamap(docs, embeddings=embeddings)


tree = topic_modeling.topic_model.get_topic_tree(hierarchical_topics)

# print(tree)
# Capture the output of the hierarchical topic tree into a StringIO buffer
connect_buffer = io.StringIO()
print(tree, file=connect_buffer)  # Assuming 'tree' holds the hierarchical topic structure
connect_tree_str = connect_buffer.getvalue()
connect_buffer.close()

# Convert special characters to HTML entities for safe embedding
connect_tree_html = html.escape(connect_tree_str)

# Generate a Matplotlib figure for the document datamap
connect_docs = df['consolidated_description'].tolist()
connect_fig = topic_modeling.topic_model.visualize_document_datamap(connect_docs, embeddings=embeddings)

# Save the figure to a BytesIO buffer as a PNG
connect_buf = BytesIO()
connect_fig.savefig(connect_buf, format='png')
connect_buf.seek(0)

# Encode the image in base64 to embed in HTML
connect_image_base64 = base64.b64encode(connect_buf.read()).decode('utf-8')
connect_buf.close()

# Ensure to close the Matplotlib figure to free memory
plt.close(connect_fig)

# Combine both the topic tree and document datamap into a single HTML document
connect_html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Connect C-Ops Comprehensive Visualization</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
        }}
        pre {{
            white-space: pre-wrap;  /* CSS for preformatted text */
            word-wrap: break-word;  /* Ensure long text wraps */
        }}
    </style>
</head>
<body>
    <h1>Connect C-Ops Topic Tree Visualization</h1>
    <pre>{connect_tree_html}</pre>
    <h1>Connect C-Ops Document Datamap Visualization</h1>
    <img src="data:image/png;base64,{connect_image_base64}" />
</body>
</html>
"""

# Write the combined HTML content to a file
with open('combined_connect_topic_visualization_set_02.html', 'w', encoding='utf-8') as connect_file:
    connect_file.write(connect_html_content)

print("All visualizations have been saved in 'combined_connect_topic_visualization_set_02.html'.")

All visualizations have been saved in 'combined_connect_topic_visualization_set_02.html'.


In [22]:
# docs = df['consolidated_description'].tolist()

# topic_modeling.topic_model.visualize_documents(docs, embeddings=embeddings)

In [23]:
# tree = topic_modeling.topic_model.get_topic_tree(hierarchical_topics)

# print(tree)

# docs = df['consolidated_description'].tolist()

# topic_modeling.topic_model.visualize_documents(docs, embeddings=embeddings)


# import datamapplot

# #from bertopic.plotting import datamap_plot

# topic_modeling.topic_model.visualize_document_datamap(docs, embeddings=embeddings)

# topic_modeling.get_topic_info()

# hierarchical_topics

# topic_modeling.topic_model.visualize_topics()

# topic_modeling.topic_model.visualize_barchart( )

# topic_modeling.topic_model.visualize_heatmap()

In [24]:
# import base64
# from io import BytesIO
# import plotly.io as pio
# import matplotlib.pyplot as plt

# # Step 1: Calculate the topic distributions on a token-level
# topic_distr, topic_token_distr = topic_modeling.topic_model.approximate_distribution(docs, calculate_tokens=True)

# # Step 2: Visualize the token-level distributions (returns a Plotly figure)
# connect_fig_token_distro = topic_modeling.topic_model.visualize_approximate_distribution(docs[1], topic_token_distr[1])

# # Step 3: Visualize the term rank (returns a Plotly figure)
# connect_fig_term_rank = topic_modeling.topic_model.visualize_term_rank()

# # Step 4: Convert the Plotly figures to HTML strings
# connect_token_distro_html = pio.to_html(connect_fig_token_distro, full_html=False)
# connect_term_rank_html = pio.to_html(connect_fig_term_rank, full_html=False)

# # Step 5: Combine the visualizations into a single HTML file
# # Assuming the previous HTML content is already created and stored in `html_content`
# connect_html_content = f"""<!DOCTYPE html>
# <html>
# <head>
#     <title>Connect C-Ops Comprehensive Visualization</title>
#     <style>
#         body {{
#             font-family: Arial, sans-serif;
#             margin: 20px;
#         }}
#         pre {{
#             white-space: pre-wrap;       /* Since CSS 2.1 */
#             word-wrap: break-word;       /* Ensure long text wraps */
#         }}
#     </style>
# </head>
# <body>
#     <h1>Connect C-Ops Topic Tree Visualization</h1>
#     <pre>{connect_tree_html}</pre>
#     <h1>Connect C-Ops Document Datamap Visualization</h1>
#     <img src="data:image/png;base64,{connect_image_base64}" />
#     <h1>Connect C-Ops Topic Word Scores</h1>
#     <img src="data:image/png;base64,{connect_barplot_base64}" />
#     <h1>Connect C-Ops Token-Level Distribution</h1>
#     {connect_token_distro_html}
#     <h1>Connect C-Ops Term Rank Visualization</h1>
#     {connect_term_rank_html}
# </body>
# </html>
# """

# # Step 6: Write the final combined HTML content to a file
# with open('connect_combined_visualization_3.html', 'w', encoding='utf-8') as connect_file:
#     connect_file.write(connect_html_content)

# print("All visualizations have been saved in 'connect_combined_visualization_3.html'.")


In [25]:
# # # docs = df['consolidated_description'].tolist()

# # # topic_modeling.topic_model.visualize_documents(docs, embeddings=embeddings)

# # # Calculate the topic distributions on a token-level
# # topic_distr, topic_token_distr = topic_modeling.topic_model.approximate_distribution(docs, calculate_tokens=True)

# # # Visualize the token-level distributions
# # df_distro = topic_modeling.topic_model.visualize_approximate_distribution(docs[1], topic_token_distr[1])
# # df_distro

# # topic_modeling.topic_model.visualize_term_rank()


# import seaborn as sns
# import matplotlib.pyplot as plt
# import pandas as pd

# # Assuming topic_modeling.topic_model is already fitted
# topic_model = topic_modeling.topic_model

# # Get the topics with their weights
# topics = topic_model.get_topics()

# # Create a DataFrame for visualization
# data = []
# for topic_num, topic in topics.items():
#     for word, weight in topic:
#         data.append({'Topic': f'Topic {topic_num}', 'Word': word, 'Weight': weight})

# connect_df_word_weight = pd.DataFrame(data)

# # Plotting
# plt.figure(figsize=(12, 10))  # Adjust the figure size as needed
# barplot = sns.barplot(x="Weight", y="Word", hue="Topic", data=connect_df_word_weight, palette="coolwarm")

# # Reduce the font size of the y-axis labels
# plt.yticks(fontsize=8)  # Adjust fontsize according to your needs

# # Optionally, if you want to adjust x-axis labels as well
# plt.xticks(fontsize=10)  # Adjust fontsize as needed

# # Title and labels
# plt.title('Topic Word Scores', fontsize=16)
# plt.xlabel('Word Importance', fontsize=12)
# plt.ylabel('Words', fontsize=12)

# # Improve the layout
# plt.tight_layout()

# # Show the plot
# plt.show()

In [26]:
# # with the original embeddings
# #!pip install datamapplot -qqq
# import datamapplot

# #from bertopic.plotting import datamap_plot

# topic_modeling.topic_model.visualize_document_datamap(docs, embeddings=embeddings)

In [27]:
topic_modeling.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,130,-1_connectdb host tpalpe0svd009_host tpalpe0sv...,"[connectdb host tpalpe0svd009, host tpalpe0svd...",[Description:pg-database:connectdb:host=tpalpe...
1,0,662,0_admin status update_receive broadsoft failur...,"[admin status update, receive broadsoft failur...",[Order has Send Order to Ordering task in acti...
2,1,109,1_healthcheck workflow error_error condition h...,"[healthcheck workflow error, error condition h...",[Description:E0SV_E0SV.Connect - ESAPNET-UI-00...
3,2,79,2_95 owner app_owner app om_apps opt_opt postg...,"[95 owner app, owner app om, apps opt, opt pos...",[Ticket Automatically Created from OMi ---- \n...
4,3,59,3_tpalpe0svw003 cpu idle_cpu idle description_...,"[tpalpe0svw003 cpu idle, cpu idle description,...",[Description:tpalpe0svw003-CPU Idle%; Conditio...
5,4,57,4_update failed task_status update failed_upda...,"[update failed task, status update failed, upd...",[Order has Recurring Call NRM Admin-Status Upd...
6,5,56,5_heap memory usage_memory usage 90_descriptio...,"[heap memory usage, memory usage 90, descripti...",[Description:Heap memory usage > 90% for at le...
7,6,40,6_disk usage source_tpse0svhlap001 dataserver ...,"[disk usage source, tpse0svhlap001 dataserver ...",[Description:tpse0svhlap001-/dataserver/logs; ...
8,7,37,7_connectdb host tpalpe0svd007_host tpalpe0svd...,"[connectdb host tpalpe0svd007, host tpalpe0svd...",[Description:pg-database:connectdb:host=tpalpe...
9,8,33,8_write service healthcheck_2372031_nts_write ...,"[write service healthcheck_2372031_nts, write ...",[Description:E0SV_E0SV.Connect - LB-tninv-nrm-...


In [28]:
# topic_modeling.topic_model.visualize_hierarchy()


In [29]:
# hierarchical_topics

In [30]:
# topic_modeling.topic_model.visualize_topics()

In [31]:
# # # Visualize hierarchy with custom labels
# # topic_model.visualize_hierarchy(custom_labels=True)

# topic_modeling.topic_model.visualize_barchart( )

In [32]:
# topic_modeling.topic_model.visualize_heatmap()

In [33]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# import numpy as np

# similarity_matrix = np.random.rand(8, 8)  # Example matrix
# np.fill_diagonal(similarity_matrix, 1)  # Ensure diagonal values are 1 (self-similarity)

# # Example list of actual labels for rows and columns (replace with your own labels)
# labels = [
#     'status update failed',
#     'healthcheck workflow error',
#     'owner app issue',
#     'heap memory usage issue',
#     'cpu idle issue',
#     'task update failed',
#     'connectdb host issue',
#     'service healthcheck'
# ]

# # Create a heatmap with annotations (values inside the boxes) and actual labels
# plt.figure(figsize=(10, 7))
# ax = sns.heatmap(similarity_matrix, annot=True, cmap="YlGnBu", fmt=".2f",
#                  xticklabels=labels, yticklabels=labels, cbar_kws={'label': 'Similarity Score'})

# # Rotate the x-axis labels for better readability
# plt.xticks(rotation=45, ha='right')

# # Set heatmap title
# ax.set_title("Similarity Matrix", fontsize=16)

# # Adjust layout to ensure proper display of labels
# plt.tight_layout()

# # Show the plot
# plt.show()

In [34]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# import pandas as pd

# # Assuming topic_modeling.topic_model is already fitted
# topic_model = topic_modeling.topic_model

# # Get the topics with their weights
# topics = topic_model.get_topics()

# # Create a DataFrame for visualization
# data = []
# for topic_num, topic in topics.items():
#     for word, weight in topic:
#         data.append({'Topic': f'Topic {topic_num}', 'Word': word, 'Weight': weight})

# connect_df_word_weight = pd.DataFrame(data)

# # Plotting
# plt.figure(figsize=(12, 10))  # Adjust the figure size as needed
# barplot = sns.barplot(x="Weight", y="Word", hue="Topic", data=connect_df_word_weight, palette="coolwarm")

# # Reduce the font size of the y-axis labels
# plt.yticks(fontsize=8)  # Adjust fontsize according to your needs

# # Optionally, if you want to adjust x-axis labels as well
# plt.xticks(fontsize=10)  # Adjust fontsize as needed

# # Title and labels
# plt.title('Topic Word Scores', fontsize=16)
# plt.xlabel('Word Importance', fontsize=12)
# plt.ylabel('Words', fontsize=12)

# # Improve the layout
# plt.tight_layout()

# # Show the plot
# plt.show()


In [35]:
# topic_modeling.topic_model.visualize_term_rank()

In [36]:
# # Calculate the topic distributions on a token-level
# topic_distr, topic_token_distr = topic_modeling.topic_model.approximate_distribution(docs, calculate_tokens=True)

# # Visualize the token-level distributions
# df_distro = topic_modeling.topic_model.visualize_approximate_distribution(docs[1], topic_token_distr[1])
# df_distro


In [37]:
import pandas as pd
import plotly.io as pio

# Step 1: Calculate the topic distributions on a token-level

docs = df['consolidated_description'].tolist()
topic_distr, topic_token_distr = topic_modeling.topic_model.approximate_distribution(docs, calculate_tokens=True)

# Step 2: Visualize the token-level distributions and get a Styler object (HTML table)
connect_df_token_distro = topic_modeling.topic_model.visualize_approximate_distribution(docs[1], topic_token_distr[1])

# Convert the Styler object to HTML
connect_token_distro_html = connect_df_token_distro.to_html()

# Step 3: Visualize the term rank (returns a Plotly figure)
connect_fig_term_rank = topic_modeling.topic_model.visualize_term_rank()

# Convert the Plotly figure to an HTML string
connect_term_rank_html = pio.to_html(connect_fig_term_rank, full_html=False)

# Step 4: Combine the visualizations into a single HTML file
connect_html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Connect C-Ops Comprehensive Visualization</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
        }}
        table {{
            border-collapse: collapse;
            width: 100%;
            margin: 20px 0;
        }}
        th, td {{
            padding: 8px;
            text-align: left;
            border: 1px solid #ddd;
        }}
        th {{
            background-color: #f2f2f2;
        }}
        .plotly-graph-div {{
            width: 90%;
            margin: auto;
        }}
    </style>
</head>
<body>
    <h1>Connect C-Ops Token-Level Distribution</h1>
    {connect_token_distro_html}  <!-- Display the token-level distribution as a table -->
    <h1>Connect C-Ops Term Rank Visualization</h1>
    {connect_term_rank_html}  <!-- Display the Plotly term rank figure -->
</body>
</html>
"""

# Step 5: Write the final combined HTML content to a file
with open('connect_token_distribution_and_term_rank_set_03.html', 'w', encoding='utf-8') as connect_file:
    connect_file.write(connect_html_content)

print("Both visualizations have been saved in 'connect_token_distribution_and_term_rank_set_03.html'.")


Both visualizations have been saved in 'connect_token_distribution_and_term_rank_set_03.html'.


In [38]:
breaking here

SyntaxError: invalid syntax (<ipython-input-38-d3aa1b68d878>, line 1)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

def elbow_method(data, max_k=10):
    sse = []
    for k in range(1, max_k):
        kmeans = KMeans(n_clusters=k, random_state=42)
        kmeans.fit(data)
        sse.append(kmeans.inertia_)  # Sum of squared distances to centroids

    plt.plot(range(1, max_k), sse, marker='o')
    plt.xlabel('Number of clusters (K)')
    plt.ylabel('SSE (Inertia)')
    plt.title('Elbow Method for Optimal K')
    plt.show()

elbow_method(embeddings)

In [ ]:
embeddings

In [ ]:
import matplotlib.pyplot as plt
import mpld3

from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

def silhouette_analysis(data, max_k=10, output_path="output_plot.html"):
    silhouette_scores = []
    for k in range(2, max_k):
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(data)
        silhouette_avg = silhouette_score(data, labels)
        silhouette_scores.append(silhouette_avg)

    fig, ax = plt.subplots()
    ax.plot(range(2, max_k), silhouette_scores, marker='o')
    ax.set_xlabel('Number of clusters (K)')
    ax.set_ylabel('Silhouette Score')
    ax.set_title('Silhouette Score for Optimal K')

    # Save the plot as an interactive HTML file
    mpld3.save_html(fig, output_path)
    print(f"Plot saved as HTML at {output_path}")

# Specify the absolute path where you want to save the HTML file
output_path = "/content/drive/MyDrive/company_project/output_plot.html"
silhouette_analysis(embeddings, output_path=output_path)


In [ ]:
pwd

In [ ]:
!pip install mpld3 -qqq

!pip install bertopic -qqq
!pip install plotly -qqq
# !pip install scipy
!pip install scipy==1.11.4  -qqq
!pip install pandas -qqq
!pip install sentence_transformers -qqq
!pip install umap-learn -qqq


Interpreting the plot:
Peak at 3 clusters: The silhouette score peaks at K=3 with a score of around 0.16. This suggests that 3 clusters could be a reasonable choice because it's where the model has the highest silhouette score, indicating well-defined clustering.

Increasing silhouette score from K=6 onward: Starting from K=6, the silhouette score steadily increases, with the highest score for K=9. However, the silhouette scores for these higher values (e.g., K=8, K=9) do not increase as dramatically as the jump from K=2 to K=3.


In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import mpld3
from sentence_transformers import SentenceTransformer
import pandas as pd
from scipy.cluster.hierarchy import fcluster


from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import numpy as np



import pandas as pd

class ConnectDataPreprocessor:
    def __init__(self, file_path):
        """
        Initialize the ConnectDataPreprocessor with the path to the CSV file.
        :param file_path: Path to the CSV file containing the data.
        """
        self.file_path = file_path

    def load_and_preprocess(self):
        """
        Load data from CSV and preprocess it by consolidating descriptions and cleaning null values.
        Returns the preprocessed DataFrame.
        """
        # Load data
        df = pd.read_csv(self.file_path)

        # Select relevant columns
        df = df[['incident_number', 'app_id', 'created_on', 'description',
                 'short_description', 'assignment_group', 'category', 'app_name',
                 'resolution_code', 'resolution_update', 'meta_tags']]

        # Combine descriptions
        df['consolidated_description'] = df['description'].fillna('') + ' ' + df['short_description'].fillna('')

        # Remove any rows with null consolidated descriptions
        df.dropna(subset=['consolidated_description'], inplace=True)

        return df

# Usage
if __name__ == "__main__":
    # Path to your data file


    file_path = "/content/drive/MyDrive/company_project/dummy_variable_data_public.csv"
    data_preprocessor = ConnectDataPreprocessor(file_path)

    # Load and preprocess data
    preprocessed_df = data_preprocessor.load_and_preprocess()
    print("Data Loaded and Preprocessed:", preprocessed_df.shape)


class ConnectTextEmbedding:
    def __init__(self, df, column_name, sentence_transformer_model='google-bert/bert-large-uncased'):

        # "dunzhang/stella_en_1.5B_v"

        """
        Initialize the TextEmbedding class to handle embedding tasks.
        :param df: DataFrame containing the text data.
        :param column_name: Name of the column containing text descriptions.
        :param sentence_transformer_model: Model to be used for sentence embeddings.
        """
        self.df = df
        self.column_name = column_name
        self.sentence_model = SentenceTransformer(sentence_transformer_model)

    def connect_encode_embeddings(self):
        """
        Encodes the descriptions into embeddings using the sentence model.
        """
        descriptions = self.df[self.column_name].tolist()
        return self.sentence_model.encode(descriptions, show_progress_bar=True)

# Usage
if __name__ == "__main__":
    # Load data
    # df = pd.read_csv("path_to_your_data.csv")  # Make sure to have the right path and data
    connect_text_embedding = ConnectTextEmbedding(preprocessed_df, "consolidated_description")

    # Generate embeddings
    connect_embeddings = connect_text_embedding.connect_encode_embeddings()

    # Now you can use 'embeddings' for any clustering or analysis tasks independently
    print("Embeddings generated:", connect_embeddings)

# Function to generate Dendrogram plot
def connect_plot_dendrogram(data, method='ward'):
    Z = linkage(data, method=method)
    fig, ax = plt.subplots(figsize=(10, 7))  # Create figure and axis
    dendrogram(Z, ax=ax)  # Create the dendrogram plot

    ax.set_title('Connect Application Dendrogram')
    ax.set_xlabel('Data Points')
    ax.set_ylabel('Distance')

    # Return the HTML for the plot
    return mpld3.fig_to_html(fig)

def connect_optimized_dendrogram(data, method='ward', metric='euclidean'):
    Z = linkage(data, method=method, metric=metric)
    fig, ax = plt.subplots(figsize=(10, 7))
    dendrogram(Z, ax=ax)
    ax.set_title(f'Connect Application Dendrogram - {method.capitalize()} Linkage')
    ax.set_xlabel('Data Points')
    ax.set_ylabel('Distance')
    return mpld3.fig_to_html(fig)


# Function to generate Silhouette Score plot
def connect_silhouette_analysis(data, max_k=10):
    connect_silhouette_scores = []
    for k in range(2, max_k):
        connect_kmeans = KMeans(n_clusters=k, random_state=42)
        connect_labels = connect_kmeans.fit_predict(data)
        connect_silhouette_avg = silhouette_score(data, connect_labels)
        connect_silhouette_scores.append(connect_silhouette_avg)

    fig, ax = plt.subplots()
    ax.plot(range(2, max_k), connect_silhouette_scores, marker='o')
    ax.set_xlabel('Number of clusters (K)')
    ax.set_ylabel('Silhouette Score')
    ax.set_title('Connect Application Silhouette Score for Optimal K')

    # Return the HTML for the plot
    return mpld3.fig_to_html(fig)

from sklearn.preprocessing import StandardScaler

def connect_optimized_silhouette(data, max_k=10, random_state=42):
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    silhouette_scores = []
    for k in range(2, max_k):
        kmeans = KMeans(n_clusters=k, random_state=random_state)
        labels = kmeans.fit_predict(data_scaled)
        score = silhouette_score(data_scaled, labels)
        silhouette_scores.append(score)
    fig, ax = plt.subplots()
    ax.plot(range(2, max_k), silhouette_scores, marker='o')
    ax.set_title('Connect Application Optimized Silhouette Scores')
    return mpld3.fig_to_html(fig)



# Function to generate Elbow Method plot
def connect_elbow_method(data, max_k=10):
    connect_sse = []
    for k in range(1, max_k):
        connect_kmeans = KMeans(n_clusters=k, random_state=42)
        connect_kmeans.fit(data)
        connect_sse.append(connect_kmeans.inertia_)  # Sum of squared distances to centroids

    fig, ax = plt.subplots()
    ax.plot(range(1, max_k), connect_sse, marker='o')
    ax.set_xlabel('Number of clusters (K)')
    ax.set_ylabel('SSE (Inertia)')
    ax.set_title('Connect Application Elbow Method for Optimal K')

    # Return the HTML for the plot
    return mpld3.fig_to_html(fig)

def connect_optimized_elbow(data, max_k=10, random_state=42):
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    sse = []
    for k in range(1, max_k):
        kmeans = KMeans(n_clusters=k, random_state=random_state)
        kmeans.fit(data_scaled)
        sse.append(kmeans.inertia_)
    fig, ax = plt.subplots()
    ax.plot(range(1, max_k), sse, marker='o')
    ax.set_title('Connect Application Optimized Elbow Method')
    return mpld3.fig_to_html(fig)



# Function to generate DBSCAN plot
def connect_dbscan_analysis(data, eps=0.5, min_samples=5):
    # Standardize the data
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # Apply DBSCAN algorithm
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(data_scaled)

    # Plot DBSCAN clustering results
    fig, ax = plt.subplots(figsize=(10, 7))

    # Plot the clustered points
    unique_labels = np.unique(labels)
    for label in unique_labels:
        label_mask = labels == label
        ax.scatter(data_scaled[label_mask, 0], data_scaled[label_mask, 1],
                   label=f'Cluster {label}' if label != -1 else 'Noise')

    # Adding title and labels
    ax.set_title('Connect Application DBSCAN Clustering')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(loc="best")

    # Return the HTML for the plot
    return mpld3.fig_to_html(fig)


# Function to fine-tune DBSCAN and return the best result
def connect_dbscan_analysis_grid_search_para_tunning(data, eps_values=[0.3, 0.5, 0.7, 1.0], min_samples_values=[5, 10, 15]):
    # Standardize the data
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    best_silhouette = -1
    best_eps = None
    best_min_samples = None
    best_labels = None

    # Grid search over eps and min_samples values
    for eps in eps_values:
        for min_samples in min_samples_values:
            dbscan = DBSCAN(eps=eps, min_samples=min_samples)
            labels = dbscan.fit_predict(data_scaled)

            # Ignore cases where DBSCAN assigns all points to noise (-1)
            if len(np.unique(labels)) > 1:
                score = silhouette_score(data_scaled, labels)
                print(f"eps: {eps}, min_samples: {min_samples}, Silhouette Score: {score:.4f}")

                if score > best_silhouette:
                    best_silhouette = score
                    best_eps = eps
                    best_min_samples = min_samples
                    best_labels = labels

    # Plot DBSCAN clustering results with the best parameters
    fig, ax = plt.subplots(figsize=(10, 7))

    # Plot the clustered points
    unique_labels = np.unique(best_labels)
    for label in unique_labels:
        label_mask = best_labels == label
        ax.scatter(data_scaled[label_mask, 0], data_scaled[label_mask, 1],
                   label=f'Cluster {label}' if label != -1 else 'Noise')

    # Adding title and labels
    ax.set_title(f'Connect Application DBSCAN Clustering (eps={best_eps}, min_samples={best_min_samples})')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(loc="best")

    # Return the HTML for the plot
    print(f'Best parameters: eps={best_eps}, min_samples={best_min_samples}, Silhouette Score={best_silhouette:.4f}')
    return mpld3.fig_to_html(fig)

# Example usage with grid search ranges
eps_values = [0.2, 0.3, 0.4, 0.5]
min_samples_values = [5, 10, 15]

# Best Value = eps = 0.2 and min_sample_values = 5

connect_dbscan_html_tunned = connect_dbscan_analysis_grid_search_para_tunning(connect_embeddings, eps_values=eps_values, min_samples_values=min_samples_values)

# Generate all three plots' HTML content
connect_dendrogram_html = connect_plot_dendrogram(connect_embeddings)
connect_silhouette_html = connect_silhouette_analysis(connect_embeddings, max_k=10)
connect_elbow_html = connect_elbow_method(connect_embeddings, max_k=10)

# Example usage
connect_dbscan_html = connect_dbscan_analysis(connect_embeddings, eps=0.3, min_samples=10)

class KMeansClustering:
    def __init__(self, n_clusters=3, random_state=42):
        """
        Initialize the K-Means clustering class.
        :param n_clusters: The number of clusters to use for K-Means.
        :param random_state: Seed for reproducibility.
        """
        self.n_clusters = n_clusters
        self.random_state = random_state

    def apply_kmeans(self, preprocessed_df, embeddings):
        """
        Apply K-Means clustering and add cluster labels to the dataframe.
        """
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=self.random_state)
        cluster_labels = kmeans.fit_predict(embeddings)

        # Add the K-Means cluster labels to the dataframe
        preprocessed_df['kmeans_cluster'] = cluster_labels
        return preprocessed_df


class HierarchicalClustering:
    def __init__(self, n_clusters=3, method='ward'):
        """
        Initialize the Hierarchical clustering class.
        :param n_clusters: The number of clusters to use.
        :param method: Linkage method ('ward', 'complete', 'average', etc.).
        """
        self.n_clusters = n_clusters
        self.method = method

    def apply_hierarchical_clustering(self, preprocessed_df, embeddings):
        """
        Apply Hierarchical clustering and add cluster labels to the dataframe.
        """
        Z = linkage(embeddings, method=self.method)

        # Generate cluster labels
        cluster_labels = fcluster(Z, self.n_clusters, criterion='maxclust')

        # Add the Hierarchical cluster labels to the dataframe
        preprocessed_df['hierarchical_cluster'] = cluster_labels
        return preprocessed_df


class DBSCANClustering:
    def __init__(self, eps=0.5, min_samples=5):
        """
        Initialize the DBSCAN clustering class.
        :param eps: The maximum distance between two samples for them to be considered as in the same neighborhood.
        :param min_samples: The minimum number of points required to form a dense region.
        """
        self.eps = eps
        self.min_samples = min_samples

    def apply_dbscan(self, preprocessed_df, embeddings):
        """
        Apply DBSCAN clustering and add cluster labels to the dataframe.
        """
        # Standardize the data
        scaler = StandardScaler()
        embeddings_scaled = scaler.fit_transform(embeddings)

        dbscan = DBSCAN(eps=self.eps, min_samples=self.min_samples)
        cluster_labels = dbscan.fit_predict(embeddings_scaled)

        # Add the DBSCAN cluster labels to the dataframe
        preprocessed_df['dbscan_cluster'] = cluster_labels
        return preprocessed_df


class OptimizedDBSCANClustering:

    def __init__(self, eps=0.2, min_samples=5):
        """
        Initialize the optimized DBSCAN clustering class.
        :param eps: The optimal eps value found through parameter tuning.
        :param min_samples: The optimal min_samples value found through parameter tuning.
        """
        self.eps = eps
        self.min_samples = min_samples

    def apply_optimized_dbscan(self, preprocessed_df, embeddings):
        """
        Apply optimized DBSCAN clustering and add cluster labels to the dataframe.
        """
        # Standardize the data
        scaler = StandardScaler()
        embeddings_scaled = scaler.fit_transform(embeddings)

        dbscan = DBSCAN(eps=self.eps, min_samples=self.min_samples)
        cluster_labels = dbscan.fit_predict(embeddings_scaled)

        # Add the optimized DBSCAN cluster labels to the dataframe
        preprocessed_df['optimized_dbscan_cluster'] = cluster_labels
        return preprocessed_df


# Example usage to apply all clustering methods
if __name__ == "__main__":
    # Load your preprocessed dataframe and embeddings
    # Assuming `preprocessed_df` and `connect_embeddings` have already been generated

    # Step 1: Apply K-Means Clustering
    kmeans_clusterer = KMeansClustering(n_clusters=3)
    preprocessed_df = kmeans_clusterer.apply_kmeans(preprocessed_df, connect_embeddings)

    # Step 2: Apply Hierarchical Clustering (Dendrogram)
    hierarchical_clusterer = HierarchicalClustering(n_clusters=3, method='ward')
    preprocessed_df = hierarchical_clusterer.apply_hierarchical_clustering(preprocessed_df, connect_embeddings)

    # Step 3: Apply DBSCAN Clustering
    # dbscan_clusterer = DBSCANClustering(eps=0.3, min_samples=10)
    # preprocessed_df = dbscan_clusterer.apply_dbscan(preprocessed_df, connect_embeddings)

    # Step 4: Apply Optimized DBSCAN Clustering
    optimized_dbscan_clusterer = OptimizedDBSCANClustering(eps=0.2, min_samples=5)
    preprocessed_df = optimized_dbscan_clusterer.apply_optimized_dbscan(preprocessed_df, connect_embeddings)

    # After applying all clustering approaches, print the resulting dataframe to check the labels
    # print(preprocessed_df.head())

    # Optionally, save the dataframe with clusters to a CSV file
    # preprocessed_df.to_csv("preprocessed_df_with_clusters.csv", index=False)


# Key observations text

connect_key_observations = """
<h2>Key Observations:</h2>
<p><strong>Dendrogram:</strong> There are three major branches at the top of the dendrogram (represented in different colors: orange, green, and red).
Cutting the dendrogram at this height would yield 3 clusters, which seems to align with the result from the silhouette analysis.
The large gap in height between the last two merges at around a distance of 60 suggests that 3 clusters is an appropriate choice here.
Cutting the dendrogram at this height would effectively separate the data into three distinct clusters.</p>

<p><strong>Silhouette Score:</strong> The silhouette score peaks at K=3, indicating that 3 clusters provide the most distinct separation among the clusters.
The score starts to drop after K=3, further confirming that 3 clusters is an optimal number for clustering.</p>

<p><strong>Elbow Method:</strong> The Elbow Method shows a clear elbow at K=3, indicating that the inertia (sum of squared distances) significantly decreases
until this point, and after K=3, the rate of decrease slows down. This supports the idea that 3 clusters are optimal for this dataset.</p>
"""

# Combine all four approaches into one HTML file with key observations
connect_output_path = "/content/drive/MyDrive/company_project/connect_combined_output_with_dbscan.html"
with open(connect_output_path, 'w') as f:
    f.write("<html><head><title>Connect Application Clustering Results</title></head><body>")
    f.write("<h1>Connect Application Dendrogram</h1>")
    f.write(connect_dendrogram_html)
    f.write("<h1>Connect Application Silhouette Score Plot</h1>")
    f.write(connect_silhouette_html)
    f.write("<h1>Connect Application Elbow Method Plot</h1>")
    f.write(connect_elbow_html)
    f.write("<h1>Connect Application DBSCAN Clustering</h1>")
    f.write(connect_dbscan_html)
    f.write(connect_key_observations)  # Add key observations at the end of the HTML file
    f.write("</body></html>")

print(f"Combined plot with DBSCAN saved as HTML at {connect_output_path}")

Key observations:
There are three major branches at the top of the dendrogram (represented in different colors: orange, green, and red). Cutting the dendrogram at this height would yield 3 clusters, which seems to align with the result from the silhouette analysis.

The large gap in height between the last two merges at around a distance of 60 suggests that 3 clusters is an appropriate choice here. Cutting the dendrogram at this height would effectively separate the data into three distinct clusters.

In [ ]:
preprocessed_df.head()

In [ ]:
import json
import psycopg2
import pandas as pd

class PostgresDatabase:
    def __init__(self, config_path):
        """
        Initialize the PostgresDatabase class by reading the JSON config file.
        :param config_path: Path to the JSON config file.
        """
        # Load the JSON config file
        with open(config_path, 'r') as file:
            self.config = json.load(file)

        # Create a connection string using the config data
        self.connection = None

    def connect(self):
        """
        Establish connection to the PostgreSQL database using the configuration provided.
        """
        try:
            self.connection = psycopg2.connect(
                dbname=self.config['dbname'],
                user=self.config['user'],
                password=self.config['password'],
                host=self.config['host'],
                port=self.config['port']
            )
            print("Connection to PostgreSQL established successfully.")
        except Exception as e:
            print(f"Error connecting to PostgreSQL: {e}")

    def close_connection(self):
        """
        Close the connection to the database.
        """
        if self.connection:
            self.connection.close()
            print("PostgreSQL connection closed.")

    def read_data(self, query=None):
        """
        Read data from PostgreSQL database and return it as a DataFrame.
        :param query: SQL query to execute. If not provided, will fetch all data from the specified table.
        :return: Pandas DataFrame containing the result set.
        """
        if query is None:
            query = f"SELECT * FROM {self.config['schema']}.{self.config['table']}"

        try:
            df = pd.read_sql(query, self.connection)
            print("Data fetched successfully.")
            return df
        except Exception as e:
            print(f"Error fetching data: {e}")
            return None

    def write_data(self, dataframe, table_name):
        """
        Write data from a Pandas DataFrame to a PostgreSQL table.
        :param dataframe: The DataFrame containing the data to write.
        :param table_name: The name of the table where data will be written.
        """
        try:
            # Establish a cursor
            cursor = self.connection.cursor()
            # Insert data row by row (can be optimized for batch inserts)
            for row in dataframe.itertuples(index=False):
                columns = ', '.join(dataframe.columns)
                values = ', '.join([f"'{str(value)}'" for value in row])
                insert_query = f"INSERT INTO {self.config['schema']}.{table_name} ({columns}) VALUES ({values})"
                cursor.execute(insert_query)

            # Commit the transaction
            self.connection.commit()
            cursor.close()
            print("Data written to PostgreSQL successfully.")
        except Exception as e:
            print(f"Error writing data to PostgreSQL: {e}")


# Example usage
if __name__ == "__main__":
    # Path to the JSON config file
    config_path = "/path/to/vz_postgrace_credential_reading.json"  # Update this path

    # Initialize the PostgresDatabase class
    db = PostgresDatabase(config_path)

    # Connect to the PostgreSQL database
    db.connect()

    # Example: Read data from the database
    query = "SELECT * FROM opsdb.ays_intake_view_1 LIMIT 5"
    data = db.read_data(query)
    print(data.head())

    # Example: Write data (assuming you have a DataFrame called df_to_write)
    # db.write_data(df_to_write, table_name='your_table')

    # Close the connection
    db.close_connection()
